In [ ]:
import sys
sys.path.append("..")

import trimesh
import numpy as np
from scipy.sparse import coo_matrix, csr_matrix, diags, vstack
from scipy.sparse.linalg import lsqr

from pathlib import Path

from animgen.core.models.model import BaseModelClass

In [ ]:
TEST_MODELS = {
    "snake_curved": Path("../generated_data/test/dec_mesh_Sea_Snake.glb"),
}

In [ ]:
snake_curved_mesh = BaseModelClass(TEST_MODELS["snake_curved"]).mesh

Rendering Multiviews...: 100%|██████████| 20/20 [00:03<00:00,  6.40it/s]


In [ ]:
def check_mesh_authenticity(mesh: trimesh.Trimesh):
    is_watertight = mesh.is_watertight
    is_winding_consistent = mesh.is_winding_consistent
    is_volume = mesh.is_volume

    return is_watertight, is_winding_consistent, is_volume


check_mesh_authenticity(snake_curved_mesh)

(True, True, True)

In [ ]:
def inspect_mesh(mesh: trimesh.Trimesh):
    print(f"Vertices: {len(mesh.vertices)}")
    print(f"Faces: {len(mesh.faces)}")

    print(f"Watertight: {mesh.is_watertight}")
    print(f"Volume: {mesh.is_volume}")
    print(f"Winding consistent: {mesh.is_winding_consistent}")

    print(f"Euler number: {mesh.euler_number}")
    print(f"Body count: {mesh.body_count}")

    print(f"Referenced vertices: {len(mesh.referenced_vertices)}")


inspect_mesh(snake_curved_mesh)

Vertices: 6431
Faces: 12862
Watertight: True
Volume: True
Winding consistent: True
Euler number: 0
Body count: 1
Referenced vertices: 6431


In [ ]:
snake_curved_mesh.show()

In [ ]:
V = snake_curved_mesh.copy().vertices
F = snake_curved_mesh.copy().faces


In [ ]:
import numpy as np


def triangle_areas(vertices, faces):
    """
    Compute the area of every triangle.

    Parameters
    ----------
    vertices : (N, 3) ndarray
    faces : (M, 3) ndarray

    Returns
    -------
    areas : (M,) ndarray
    """
    v0 = vertices[faces[:, 0]]
    v1 = vertices[faces[:, 1]]
    v2 = vertices[faces[:, 2]]

    cross = np.cross(v1 - v0, v2 - v0)
    return 0.5 * np.linalg.norm(cross, axis=1)


def vertex_areas(vertices, faces):
    """
    Compute one-third barycentric area for every vertex.

    Parameters
    ----------
    vertices : (N, 3) ndarray
    faces : (M, 3) ndarray

    Returns
    -------
    areas : (N,) ndarray
    """
    face_areas = triangle_areas(vertices, faces)

    vertex_areas = np.zeros(len(vertices), dtype=np.float64)

    np.add.at(vertex_areas, faces[:, 0], face_areas / 3.0)
    np.add.at(vertex_areas, faces[:, 1], face_areas / 3.0)
    np.add.at(vertex_areas, faces[:, 2], face_areas / 3.0)

    return vertex_areas

In [ ]:
# TESTING

face_areas = triangle_areas(V, F)
vertex_area = vertex_areas(V, F)

print(f"Surface area (mesh):     {snake_curved_mesh.area:.6f}")
print(f"Surface area (triangles): {face_areas.sum():.6f}")
print(f"Surface area (vertices):  {vertex_area.sum():.6f}")

Surface area (mesh):     0.670597
Surface area (triangles): 0.670597
Surface area (vertices):  0.670597


In [ ]:
def cotangent(u, v):
    """
    Compute cot(theta) between vectors u and v.

    Parameters
    ----------
    u : (..., 3)
    v : (..., 3)

    Returns
    -------
    cot : (...)
    """
    cross = np.cross(u, v)
    cross_norm = np.linalg.norm(cross, axis=-1)

    dot = np.sum(u * v, axis=-1)

    # Clamp to avoid numerical explosions in degenerate triangles
    cot_val = dot / np.maximum(cross_norm, 1e-12)
    return np.clip(cot_val, -1e4, 1e4)


In [ ]:
v0 = V[F[:, 0]]
v1 = V[F[:, 1]]
v2 = V[F[:, 2]]

cot0 = cotangent(v1 - v0, v2 - v0)
cot1 = cotangent(v2 - v1, v0 - v1)

cot2 = cotangent(v0 - v2, v1 - v2)

print(cot0.min(), cot0.max())
print(cot1.min(), cot1.max())
print(cot2.min(), cot2.max())

-3.8891898535217293 24.98001777689177
-4.810256978903582 10.765855830633138
-3.298017588193293 11.78327145553472


In [ ]:
def cotangent_laplacian(vertices, faces):
    n = len(vertices)

    v0 = vertices[faces[:, 0]]
    v1 = vertices[faces[:, 1]]
    v2 = vertices[faces[:, 2]]

    cot0 = cotangent(v1 - v0, v2 - v0)
    cot1 = cotangent(v2 - v1, v0 - v1)
    cot2 = cotangent(v0 - v2, v1 - v2)

    i = faces[:, 0]
    j = faces[:, 1]
    k = faces[:, 2]

    rows = np.concatenate([
        i, j,
        j, k,
        k, i
    ])

    cols = np.concatenate([
        j, i,
        k, j,
        i, k
    ])

    data = 0.5 * np.concatenate([
        cot2, cot2,
        cot0, cot0,
        cot1, cot1
    ])

    L = coo_matrix((data, (rows, cols)), shape=(n, n)).tocsr()

    diag = np.asarray(L.sum(axis=1)).ravel()

    L = L - diags(diag)

    return L

In [ ]:
L = cotangent_laplacian(V, F)

print(L.shape)
print(L.nnz)

print(np.abs(L.sum(axis=1)).max())

(6431, 6431)
45017
3.552713678800501e-15


In [ ]:
from scipy.sparse.linalg import spsolve

def contraction_step(
    vertices,
    faces,
    WL,
    WH,
):
    """
    Perform one mesh contraction iteration.

    Parameters
    ----------
    vertices : (N, 3)
    faces : (M, 3)
    WL : (N,)
        Laplacian weights.
    WH : (N,)
        Positional attraction weights.

    Returns
    -------
    new_vertices : (N, 3)
    """

    # Cotangent Laplacian
    L = cotangent_laplacian(vertices, faces)

    # Weight matrices
    WL2 = diags(WL ** 2)
    WH2 = diags(WH ** 2)

    # Normal equations: (L.T @ WL2 @ L + WH2) V' = WH2 @ V
    M = L.T @ WL2 @ L + WH2
    RHS = WH2 @ vertices

    return spsolve(M, RHS)


In [ ]:
N = len(V)

WL = np.ones(N)
WH = np.ones(N)

new_V = contraction_step(
    V,
    F,
    WL,
    WH,
)

In [ ]:
contracted = snake_curved_mesh.copy()
contracted.vertices = new_V

contracted.show()

In [ ]:
disp = np.linalg.norm(new_V - V, axis=1)

print("Mean displacement:", disp.mean())
print("Max displacement :", disp.max())

Mean displacement: 0.0010646185405761655
Max displacement : 0.004929812865156907


In [ ]:
for weight in [1, 2, 5, 10, 20, 50, 100]:
    new_V = contraction_step(V, F,
                             WL=np.full(len(V), weight),
                             WH=np.ones(len(V)))

    disp = np.linalg.norm(new_V - V, axis=1).mean()

    print(weight, disp)

1 0.0010646185405761655
2 0.0020413611453886714


/media/kahnsvaer/Datasets/PsnlProjects/AnimationGenerationGSoC/.venv/lib/python3.11/site-packages/scipy/sparse/_construct.py:543: FutureWarning: Input has data type int64, but the output has been cast to float64.  In the future, the output data type will match the input. To avoid this warning, set the `dtype` parameter to `None` to have the output dtype match the input, or set it to the desired output data type.
Note: In Python 3.11, this warning can be generated by a call of scipy.sparse.diags(), but the code indicated in the warning message will refer to an internal call of scipy.sparse.diags_array(). If that happens, check your code for the use of diags().
  A = diags_array(diagonals, offsets=offsets, shape=shape, dtype=dtype)


5 0.0053106013397834426
10 0.01202879346427442
20 0.02308751457292876
50 0.034302842297461306
100 0.03815267628334591


In [ ]:
max_iters = 15

V = snake_curved_mesh.vertices.copy()
F = snake_curved_mesh.faces

N = len(V)

face_areas = triangle_areas(V, F)
A = face_areas.mean()

WL = np.full(N, 1e-3 / np.sqrt(A))
WH = np.ones(N)

A0 = vertex_areas(V, F)
vol0 = snake_curved_mesh.volume

contracted_meshes = []

for i in range(max_iters):

    V_prev = V.copy()

    # One contraction iteration
    V = contraction_step(V, F, WL, WH)

    # Update weights
    At = vertex_areas(V, F)

    WL *= 2.0
    WH = np.sqrt(A0 / np.maximum(At, 1e-12))

    # Save current mesh
    contracted = snake_curved_mesh.copy()
    contracted.vertices = V.copy()

    contracted_meshes.append(contracted)

    vol = contracted.volume
    print(
        f"Iter {i:2d}",
        f"Volume: {vol:.6f}",
        f"WL: {WL.mean():.2f}",
        f"WH: {WH.mean():.2f}",
        f"Disp: {np.linalg.norm(V - V_prev, axis=1).mean():.6f}",
    )

    if abs(vol / vol0) < 1e-6:
        print(f"Early stopping at iteration {i} as volume ratio dropped below 1e-6.")
        break


Iter  0 Volume: 0.011696 WL: 0.28 WH: 1.01 Disp: 0.000111
Iter  1 Volume: 0.011654 WL: 0.55 WH: 1.02 Disp: 0.000242
Iter  2 Volume: 0.011552 WL: 1.11 WH: 1.07 Disp: 0.000454
Iter  3 Volume: 0.011269 WL: 2.22 WH: 1.19 Disp: 0.000771
Iter  4 Volume: 0.010376 WL: 4.43 WH: 1.40 Disp: 0.001476
Iter  5 Volume: 0.008030 WL: 8.86 WH: 1.83 Disp: 0.003539
Iter  6 Volume: 0.004154 WL: 17.73 WH: 5.83 Disp: 0.007674
Iter  7 Volume: 0.001095 WL: 35.45 WH: 12.55 Disp: 0.010885
Iter  8 Volume: 0.000102 WL: 70.91 WH: 26.49 Disp: 0.010834
Iter  9 Volume: 0.000004 WL: 141.82 WH: 66.35 Disp: 0.016639
Iter 10 Volume: 0.000001 WL: 283.63 WH: 132.99 Disp: 0.027614
Iter 11 Volume: 0.000004 WL: 567.26 WH: 271.14 Disp: 0.040998
Iter 12 Volume: 0.000000 WL: 1134.52 WH: 278.99 Disp: 0.058631
Iter 13 Volume: -0.000000 WL: 2269.05 WH: 240.02 Disp: 0.079055
Iter 14 Volume: 0.000000 WL: 4538.09 WH: 517.63 Disp: 0.116172
Early stopping at iteration 14 as volume ratio dropped below 1e-6.


In [ ]:
contracted_meshes[8].show()

In [ ]:
    contracted_meshes[0].show()